In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("CSIT554_FinalProject")
    .getOrCreate()
)

spark.range(5).show()


+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



Task 1: Data Collection

In [2]:
import pandas as pd

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

cols = [
    "age","workclass","fnlwgt","education","education_num",
    "marital_status","occupation","relationship","race","sex",
    "capital_gain","capital_loss","hours_per_week","native_country","income"
]

pdf = pd.read_csv(url, header=None, names=cols, skipinitialspace=True)
df = spark.createDataFrame(pdf)

display(df)

df.show(5)
print("Total rows:", df.count())


DataFrame[age: bigint, workclass: string, fnlwgt: bigint, education: string, education_num: bigint, marital_status: string, occupation: string, relationship: string, race: string, sex: string, capital_gain: bigint, capital_loss: bigint, hours_per_week: bigint, native_country: string, income: string]

+---+----------------+------+---------+-------------+------------------+-----------------+-------------+-----+------+------------+------------+--------------+--------------+------+
|age|       workclass|fnlwgt|education|education_num|    marital_status|       occupation| relationship| race|   sex|capital_gain|capital_loss|hours_per_week|native_country|income|
+---+----------------+------+---------+-------------+------------------+-----------------+-------------+-----+------+------------+------------+--------------+--------------+------+
| 39|       State-gov| 77516|Bachelors|           13|     Never-married|     Adm-clerical|Not-in-family|White|  Male|        2174|           0|            40| United-States| <=50K|
| 50|Self-emp-not-inc| 83311|Bachelors|           13|Married-civ-spouse|  Exec-managerial|      Husband|White|  Male|           0|           0|            13| United-States| <=50K|
| 38|         Private|215646|  HS-grad|            9|          Divorced|Handlers-cleaners|Not-i

Task 2: Data Cleaning

In [3]:
from pyspark.sql.functions import col, trim

for c in df.columns:
    df = df.withColumn(c, trim(col(c)))

df = df.replace("?", None).dropna()

numeric_cols = ["age","fnlwgt","education_num","capital_gain","capital_loss","hours_per_week"]
for c in numeric_cols:
    df = df.withColumn(c, col(c).cast("int"))

display(df)


DataFrame[age: int, workclass: string, fnlwgt: int, education: string, education_num: int, marital_status: string, occupation: string, relationship: string, race: string, sex: string, capital_gain: int, capital_loss: int, hours_per_week: int, native_country: string, income: string]

In [4]:
df.show(5)
print("Total rows after cleaning:", df.count())

+---+----------------+------+---------+-------------+------------------+-----------------+-------------+-----+------+------------+------------+--------------+--------------+------+
|age|       workclass|fnlwgt|education|education_num|    marital_status|       occupation| relationship| race|   sex|capital_gain|capital_loss|hours_per_week|native_country|income|
+---+----------------+------+---------+-------------+------------------+-----------------+-------------+-----+------+------------+------------+--------------+--------------+------+
| 39|       State-gov| 77516|Bachelors|           13|     Never-married|     Adm-clerical|Not-in-family|White|  Male|        2174|           0|            40| United-States| <=50K|
| 50|Self-emp-not-inc| 83311|Bachelors|           13|Married-civ-spouse|  Exec-managerial|      Husband|White|  Male|           0|           0|            13| United-States| <=50K|
| 38|         Private|215646|  HS-grad|            9|          Divorced|Handlers-cleaners|Not-i

Task 3: Feature Engineering

In [5]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml import Pipeline

label_indexer = StringIndexer(
    inputCol="income",
    outputCol="label"
)

categorical_cols = [
    "workclass", "education", "marital_status",
    "occupation", "relationship", "race",
    "sex", "native_country"
]

# StringIndexers for categorical columns
indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=c + "_idx",
        handleInvalid="skip"
    )
    for c in categorical_cols
]

# OneHotEncoders for indexed columns
encoders = [
    OneHotEncoder(
        inputCol=c + "_idx",
        outputCol=c + "_ohe"
    )
    for c in categorical_cols
]

# Numeric feature columns
numeric_cols = [
    "age", "fnlwgt", "education_num",
    "capital_gain", "capital_loss", "hours_per_week"
]

# VectorAssembler
assembler = VectorAssembler(
    inputCols=numeric_cols + [c + "_ohe" for c in categorical_cols],
    outputCol="features"
)

# Build and run pipeline
pipeline = Pipeline(
    stages=indexers + encoders + [label_indexer, assembler]
)

fe_df = pipeline.fit(df).transform(df)

# Final dataset (features + label)
final_df = fe_df.select("features", "label")

# Train / Test split (80 / 20)
train_df, test_df = final_df.randomSplit([0.8, 0.2], seed=42)

display(final_df)

final_df.show(5)
print("Training rows:", train_df.count())
print("Testing rows:", test_df.count())


DataFrame[features: vector, label: double]

+--------------------+-----+
|            features|label|
+--------------------+-----+
|(96,[0,1,2,3,5,9,...|  0.0|
|(96,[0,1,2,5,7,14...|  0.0|
|(96,[0,1,2,5,6,12...|  0.0|
|(96,[0,1,2,5,6,17...|  0.0|
|(96,[0,1,2,5,6,14...|  0.0|
+--------------------+-----+
only showing top 5 rows
Training rows: 24254
Testing rows: 5908


Task 4: Model Training

In [6]:
from pyspark.ml.classification import LogisticRegression, GBTClassifier

# Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=100
)

lr_model = lr.fit(train_df)

# Model summary
print("Logistic Regression Model:")
print("  maxIter =", lr_model.getMaxIter())
print("  regParam =", lr_model.getRegParam())
print("  elasticNetParam =", lr_model.getElasticNetParam())


# Gradient Boosted Tree model
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label",
    maxIter=20,
    maxDepth=5
)

gbt_model = gbt.fit(train_df)

#Model summary
print("\nGradient Boosted Tree Model:")
print("  maxIter =", gbt_model.getMaxIter())
print("  maxDepth =", gbt_model.getMaxDepth())



Logistic Regression Model:
  maxIter = 100
  regParam = 0.0
  elasticNetParam = 0.0

Gradient Boosted Tree Model:
  maxIter = 20
  maxDepth = 5


Task 5: Hyperparameter Tuning and Evaluation

In [7]:
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.classification import LogisticRegression, GBTClassifier

# Evaluator
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# Logistic Regression
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

lr_param_grid = (
    ParamGridBuilder()
    .addGrid(lr.regParam, [0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5])
    .build()
)

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_param_grid,
    evaluator=evaluator,
    numFolds=5
)

lr_cv_model = lr_cv.fit(train_df)

lr_best_auc = evaluator.evaluate(
    lr_cv_model.bestModel.transform(train_df)
)

print("Best Logistic Regression AUC (Training):", lr_best_auc)

# Gradient Boosted Tree – Grid Search + CV
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label"
)

gbt_param_grid = (
    ParamGridBuilder()
    .addGrid(gbt.maxDepth, [3, 5])
    .addGrid(gbt.maxIter, [10, 20])
    .build()
)

gbt_cv = CrossValidator(
    estimator=gbt,
    estimatorParamMaps=gbt_param_grid,
    evaluator=evaluator,
    numFolds=5
)

gbt_cv_model = gbt_cv.fit(train_df)

gbt_best_auc = evaluator.evaluate(
    gbt_cv_model.bestModel.transform(train_df)
)

print("Best GBT AUC (Training):", gbt_best_auc)





Best Logistic Regression AUC (Training): 0.9040114408253694
Best GBT AUC (Training): 0.9183811675478231


Task 6: Prediction and Final Evaluation

In [8]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Evaluator
evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    metricName="areaUnderROC"
)

# Logistic Regression – Test AUC
lr_test_predictions = lr_cv_model.bestModel.transform(test_df)
lr_test_auc = evaluator.evaluate(lr_test_predictions)

print("Logistic Regression Test AUC:", lr_test_auc)

# Gradient Boosted Tree – Test AUC
gbt_test_predictions = gbt_cv_model.bestModel.transform(test_df)
gbt_test_auc = evaluator.evaluate(gbt_test_predictions)

print("Gradient Boosted Tree Test AUC:", gbt_test_auc)



Logistic Regression Test AUC: 0.9001085604119466
Gradient Boosted Tree Test AUC: 0.9052227567261266
